# ViFinQA — Colab: prepare corpus and CPU indexes
Chạy tuần tự. Cell tải dữ liệu cần Internet. Có thể thay `PROJECT_SOURCE` bằng thư mục repo trên Google Drive.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
WORK_ROOT = Path("/content/vifinqa-work")
PROJECT = WORK_ROOT / "AI-Financial-Data-Assistant"
DRIVE_ARTIFACTS = Path("/content/drive/MyDrive/vifinqa-artifacts")
print(platform.platform(), "python", sys.version, "cpus", os.cpu_count())

In [ ]:
# Optional but recommended for persistence.
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [ ]:
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "huggingface-hub",
        "pyarrow",
        "bm25s",
        "PyStemmer",
        "lxml",
        "ftfy",
        "unidecode",
        "rapidfuzz",
    ],
    check=True,
)
os.chdir(PROJECT)
print("project:", PROJECT)

In [ ]:
from huggingface_hub import snapshot_download

DATA_ROOT = PROJECT / "data/raw/ViFinQA"
if not (DATA_ROOT / "questions/questions.jsonl").exists():
    snapshot_download(repo_id="AIGuruTinix/ViFinQA", repo_type="dataset", local_dir=DATA_ROOT)
assert (DATA_ROOT / "questions/questions.jsonl").is_file()
print("dataset:", DATA_ROOT)

In [ ]:
subprocess.run([sys.executable, "scripts/00_audit_dataset.py"], check=True)
subprocess.run(
    [sys.executable, "scripts/10_build_manifest.py", "--workers", str(min(6, os.cpu_count() or 1))],
    check=True,
)
subprocess.run([sys.executable, "scripts/20_build_bm25.py"], check=True)
subprocess.run(
    [sys.executable, "scripts/30_retrieve_questions.py", "--candidate-k", "2000"], check=True
)
subprocess.run([sys.executable, "scripts/32_validate_retrieval.py"], check=True)

In [ ]:
# Persist only reproducible artefacts; the raw corpus can be downloaded again.
for relative in [
    "data/interim/dataset_audit.json",
    "data/processed/table_manifest.jsonl",
    "data/processed/table_manifest.parquet",
    "data/processed/table_manifest.metadata.json",
    "data/index/bm25",
    "outputs/retrieval.jsonl",
    "outputs/retrieval_qc.json",
]:
    source = PROJECT / relative
    target = DRIVE_ARTIFACTS / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, target, dirs_exist_ok=True)
    elif source.exists():
        shutil.copy2(source, target)
print("saved to", DRIVE_ARTIFACTS)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT, check=True)
print("DONE")